# Plan 2: Multi-Task Supervoxel GNN — Regression + Edge + Uncertainty + Explainability**Research Question:** Can a single supervoxel encoder simultaneously learn tumor proportion regression, boundary classification, and segmentation uncertainty?## Architecture (~919K parameters, ~3.7 MB)```Patch Tensor (16×19) → PatchEmbedder (Transformer 3L×4H) → 128-dim    → + Laplacian PE (8-dim) → GraphEncoder (GATv2 3L×4H) → 128-dim    ├→ RegressionHead (ensemble ×4) → y_reg ∈ [0,1]  (Task 1)    ├→ EdgeHead → 10-class boundary type               (Task 2)    └→ UncertaintyHead → seg error probability          (Task 3)```**Loss:** Kendall et al. (2018) uncertainty-weighted: L = Σ (1/2σᵢ²)·Lᵢ + log(σᵢ)**Explainability:** 5-level traces (graph, patch, regression refinement, task divergence, uncertainty-driven)

In [ ]:
!pip install -q "numpy<2" torch==2.1.2 torchvision==0.16.2 --index-url https://download.pytorch.org/whl/cu118
!pip install -q torch-geometric scikit-image nibabel segmentation-models-pytorch

## Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import random
import time
from pathlib import Path

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

SHARED = {"seed": SEED, "device": DEVICE, "img_size": 224, "checkpoint_dir": Path("checkpoints")}

GRAPH = {
    "data_root": Path("/kaggle/input/brats2023-gli/BraTS-GLI"),
    "modalities": ["t1n", "t1c", "t2w", "t2f"],
    "slic_modality": "t1n",
    "num_classes": 4,
    "class_names": {0: "BG", 1: "NCR", 2: "ED", 3: "ET"},
    "n_segments": 1000, "compactness": 0.1, "min_sv_volume": 20,
    "tau": 0.15,
    "n_patch": 4, "patch_neighbors": 16, "knn_k": 8,
    "embed_dim": 128,
    "transformer_layers": 3, "transformer_heads": 4,
    "gat_layers": 3, "gat_heads": 4,
    "laplacian_pe_dim": 8,
    "regression_ensemble_size": 4,
    "n_boundary_types": 10,
    "epochs": 120, "lr": 3e-4, "weight_decay": 0.01,
    "batch_size": 2, "accum_steps": 4, "eval_every": 5, "num_folds": 5,
    "init_log_var_reg": 0.0, "init_log_var_edge": 0.0, "init_log_var_unc": 0.0,
    "checkpoint": Path("checkpoints/graph_plan2.pth"),
    "cache_dir": Path("graph_cache"),
}

print(f"Device: {DEVICE}")
print(f"Config: embed_dim={GRAPH['embed_dim']}, ensemble_K={GRAPH['regression_ensemble_size']}")

## Phase 3: Segmentation Model (DeepLabV3+)Train DeepLabV3+ on BraTS slices, then export 3D probability volumes as seg priors for the GNN.

In [ ]:
import cv2
import nibabel as nib
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import segmentation_models_pytorch as smp

SEG_CONFIG = {
    "data_root": GRAPH["data_root"],
    "modalities": ["t1n", "t1c", "t2w", "t2f"],
    "num_classes": 4,
    "batch_size": 8, "num_workers": 2,
    "lr": 1e-4, "weight_decay": 1e-4,
    "epochs": 30, "accum_steps": 4, "eval_every": 5,
    "min_tumor_pixels": 50,
    "output_dir": Path("brats_outputs"),
    "checkpoint": Path("checkpoints/segmentation_brats.pth"),
}
IMG_SIZE = 224
print("Segmentation config loaded")

### Data Preparation + Training

In [ ]:
def seg_load_volume(p): return nib.load(str(p)).get_fdata().astype(np.float32)

def seg_zscore(v):
    out = v.copy(); mask = out > 0
    if mask.sum() == 0: return out
    m, s = out[mask].mean(), out[mask].std()
    if s < 1e-8: return out
    out[mask] = (out[mask] - m) / s; return out

def seg_discover(data_dir):
    valid = []
    for d in sorted(Path(data_dir).iterdir()):
        if not d.is_dir(): continue
        cid, files, ok = d.name, {}, True
        for key in [*SEG_CONFIG["modalities"], "seg"]:
            for ext in [".nii", ".nii.gz"]:
                p = d / f"{cid}-{key}{ext}"
                if p.exists(): files[key] = p; break
            if key not in files: ok = False; break
        if ok: valid.append({"case_id": cid, "files": files})
    return valid

def extract_slices(case, cache_dir):
    prefix = f"{case['case_id']}_s"
    existing = sorted(cache_dir.glob(f"{prefix}*_img.npy"))
    if existing:
        saved = []
        for ip in existing:
            mp = cache_dir / (ip.stem.replace("_img", "_mask") + ".npy")
            if mp.exists():
                si = int(ip.stem.split("_s")[-1].replace("_img", ""))
                saved.append({"img_path": str(ip), "mask_path": str(mp), "case_id": case["case_id"], "slice_idx": si})
        if saved: return saved
    seg_vol = seg_load_volume(case["files"]["seg"])
    mvols = {m: seg_zscore(seg_load_volume(case["files"][m])) for m in SEG_CONFIG["modalities"]}
    saved = []
    for s in range(seg_vol.shape[2]):
        if (seg_vol[:,:,s] > 0).sum() < SEG_CONFIG["min_tumor_pixels"]: continue
        chs = [cv2.resize(mvols[m][:,:,s], (IMG_SIZE,IMG_SIZE), interpolation=cv2.INTER_LINEAR) for m in SEG_CONFIG["modalities"]]
        mask = cv2.resize(seg_vol[:,:,s], (IMG_SIZE,IMG_SIZE), interpolation=cv2.INTER_NEAREST).astype(np.int64)
        fn = f"{case['case_id']}_s{s:03d}"
        np.save(str(cache_dir/f"{fn}_img.npy"), np.stack(chs,0).astype(np.float32))
        np.save(str(cache_dir/f"{fn}_mask.npy"), mask)
        saved.append({"img_path": str(cache_dir/f"{fn}_img.npy"), "mask_path": str(cache_dir/f"{fn}_mask.npy"), "case_id": case["case_id"], "slice_idx": s})
    return saved

class SliceDS(Dataset):
    def __init__(self, meta, aug=False): self.meta, self.aug = meta, aug
    def __len__(self): return len(self.meta)
    def __getitem__(self, i):
        m = self.meta[i]; img, mask = np.load(m["img_path"]).copy(), np.load(m["mask_path"]).copy()
        if self.aug:
            if random.random()>0.5: img, mask = img[:,:,::-1].copy(), mask[:,::-1].copy()
            if random.random()>0.5: img, mask = img[:,::-1,:].copy(), mask[::-1,:].copy()
            k = random.randint(0,3)
            if k>0: img, mask = np.rot90(img,k,axes=(1,2)).copy(), np.rot90(mask,k,axes=(0,1)).copy()
        return torch.from_numpy(img).float(), torch.from_numpy(mask).long()

# Prepare
cache_dir = SEG_CONFIG["output_dir"] / "slice_cache"; cache_dir.mkdir(parents=True, exist_ok=True)
all_seg_cases = seg_discover(SEG_CONFIG["data_root"])
SEG_FRACTION = 0.25
seg_cases = all_seg_cases[:int(len(all_seg_cases) * SEG_FRACTION)]
print(f"Using {len(seg_cases)}/{len(all_seg_cases)} cases ({SEG_FRACTION:.0%})")

all_meta = []
for i, c in enumerate(seg_cases):
    all_meta.extend(extract_slices(c, cache_dir))
    if (i+1)%50==0 or i==0 or i==len(seg_cases)-1: print(f"  [{i+1}/{len(seg_cases)}] slices: {len(all_meta)}")

uids = sorted(set(s["case_id"] for s in all_meta))
tr_ids, tmp = train_test_split(uids, test_size=0.30, random_state=SEED)
va_ids, te_ids = train_test_split(tmp, test_size=0.50, random_state=SEED)
tr_m = [s for s in all_meta if s["case_id"] in set(tr_ids)]
va_m = [s for s in all_meta if s["case_id"] in set(va_ids)]
print(f"Split: train={len(tr_m)}, val={len(va_m)}")

tr_dl = DataLoader(SliceDS(tr_m, aug=True), batch_size=SEG_CONFIG["batch_size"], shuffle=True, num_workers=2, pin_memory=True)
va_dl = DataLoader(SliceDS(va_m), batch_size=SEG_CONFIG["batch_size"], num_workers=2, pin_memory=True)

# Model
seg_model = smp.DeepLabV3Plus(encoder_name="efficientnet-b4", encoder_weights="imagenet", in_channels=4, classes=4)
fc = seg_model.encoder._conv_stem
if fc.in_channels != 4:
    nc = nn.Conv2d(4, fc.out_channels, kernel_size=fc.kernel_size, stride=fc.stride, padding=fc.padding, bias=fc.bias is not None)
    with torch.no_grad(): nc.weight[:,:3]=fc.weight; nc.weight[:,3]=fc.weight[:,0]
    seg_model.encoder._conv_stem = nc
seg_model = seg_model.to(DEVICE)
print(f"DeepLabV3+ params: {sum(p.numel() for p in seg_model.parameters()):,}")

# DiceCE Loss
class DiceCELoss(nn.Module):
    def __init__(self, nc=4, sm=1e-5):
        super().__init__(); self.nc, self.sm, self.ce = nc, sm, nn.CrossEntropyLoss()
    def forward(self, logits, tgt):
        ce = self.ce(logits, tgt)
        ps = F.softmax(logits, dim=1); oh = F.one_hot(tgt, self.nc).permute(0,3,1,2).float()
        inter = (ps*oh).sum(dim=(0,2,3)); card = ps.sum(dim=(0,2,3))+oh.sum(dim=(0,2,3))
        return ce + 1.0 - ((2*inter+self.sm)/(card+self.sm))[1:].mean()

criterion = DiceCELoss()
seg_opt = torch.optim.AdamW(seg_model.parameters(), lr=SEG_CONFIG["lr"], weight_decay=SEG_CONFIG["weight_decay"])
seg_sched = torch.optim.lr_scheduler.OneCycleLR(seg_opt, max_lr=SEG_CONFIG["lr"],
    steps_per_epoch=len(tr_dl)//SEG_CONFIG["accum_steps"]+1, epochs=SEG_CONFIG["epochs"])

best_dice = 0.0; Path("checkpoints").mkdir(exist_ok=True)
print(f"\nTraining seg for {SEG_CONFIG['epochs']} epochs...")
for ep in range(SEG_CONFIG["epochs"]):
    seg_model.train(); el = 0.0; seg_opt.zero_grad()
    for bi, (imgs, masks) in enumerate(tr_dl):
        loss = criterion(seg_model(imgs.to(DEVICE)), masks.to(DEVICE)) / SEG_CONFIG["accum_steps"]
        loss.backward()
        if (bi+1)%SEG_CONFIG["accum_steps"]==0 or (bi+1)==len(tr_dl):
            nn.utils.clip_grad_norm_(seg_model.parameters(), 1.0); seg_opt.step(); seg_sched.step(); seg_opt.zero_grad()
        el += loss.item()*SEG_CONFIG["accum_steps"]
    if (ep+1)%SEG_CONFIG["eval_every"]==0 or ep==0:
        seg_model.eval(); vd = {1:[],2:[],3:[]}
        with torch.no_grad():
            for imgs, masks in va_dl:
                preds = seg_model(imgs.to(DEVICE)).argmax(1)
                for b in range(preds.size(0)):
                    for c in range(1,4):
                        p,g = (preds[b]==c).float(), (masks[b].to(DEVICE)==c).float()
                        vd[c].append(((2*(p*g).sum()+1e-5)/(p.sum()+g.sum()+1e-5)).item())
        md = np.mean([np.mean(vd[c]) for c in vd])
        print(f"Ep {ep+1:3d} | loss={el/len(tr_dl):.4f} | NCR={np.mean(vd[1]):.3f} ED={np.mean(vd[2]):.3f} ET={np.mean(vd[3]):.3f} | mean={md:.3f}")
        if md > best_dice: best_dice = md; torch.save(seg_model.state_dict(), str(SEG_CONFIG["checkpoint"])); print("  -> saved")

seg_model.load_state_dict(torch.load(str(SEG_CONFIG["checkpoint"]), map_location=DEVICE, weights_only=True))
print(f"\nBest seg Dice: {best_dice:.4f}")

### Export Probability Volumes

In [ ]:
seg_prob_dir = SEG_CONFIG["output_dir"] / "seg_probs"
seg_prob_dir.mkdir(parents=True, exist_ok=True)
seg_model.eval()
export_cases = seg_discover(SEG_CONFIG["data_root"])[:int(len(seg_discover(SEG_CONFIG["data_root"])) * 0.25)]
exported, cached = 0, 0

with torch.no_grad():
    for i, case in enumerate(export_cases):
        pp = seg_prob_dir / f"{case['case_id']}_seg_probs.npy"
        if pp.exists(): cached += 1; continue
        mvols = {m: seg_zscore(seg_load_volume(case["files"][m])) for m in SEG_CONFIG["modalities"]}
        H,W,D = mvols[SEG_CONFIG["modalities"][0]].shape
        probs = np.zeros((4,H,W,D), dtype=np.float32)
        for s in range(D):
            chs = [cv2.resize(mvols[m][:,:,s], (IMG_SIZE,IMG_SIZE), interpolation=cv2.INTER_LINEAR) for m in SEG_CONFIG["modalities"]]
            inp = torch.from_numpy(np.stack(chs)).unsqueeze(0).float().to(DEVICE)
            p = F.softmax(seg_model(inp), dim=1).squeeze(0).cpu().numpy()
            for c in range(4): probs[c,:,:,s] = cv2.resize(p[c], (W,H), interpolation=cv2.INTER_LINEAR)
        np.save(str(pp), probs); del mvols, probs; exported += 1
        if (i+1)%20==0 or i==0 or (i+1)==len(export_cases): print(f"  [{i+1}/{len(export_cases)}] exported={exported}, cached={cached}")

print(f"\nDone: {exported} new + {cached} cached → {seg_prob_dir}")

## Supervoxel Preprocessing (Steps 1–7)Same pipeline as Plan 1: 3D SLIC → background pruning → GT → seg priors → patches → kNN → edge GT.

In [ ]:
import nibabel as nib
from skimage.segmentation import slic
from sklearn.cluster import KMeans
from scipy.spatial import cKDTree



# ── Data Loading (native resolution) ─────────────────────────────────


def discover_cases(data_dir):
    data_dir = Path(data_dir)
    cases = sorted([d for d in data_dir.iterdir() if d.is_dir()])
    valid = []

    for case_dir in cases:
        case_id = case_dir.name
        files = {}
        all_found = True

        for key in [*GRAPH["modalities"], "seg"]:
            # Try both .nii and .nii.gz
            nii = case_dir / f"{case_id}-{key}.nii"
            nii_gz = case_dir / f"{case_id}-{key}.nii.gz"
            if nii.exists():
                files[key] = nii
            elif nii_gz.exists():
                files[key] = nii_gz
            else:
                all_found = False
                break

        if all_found:
            valid.append({"case_id": case_id, "files": files})

    return valid


def load_volume(nii_path):
    """Load a NIfTI volume at native resolution as float32."""
    return nib.load(str(nii_path)).get_fdata().astype(np.float32)


def zscore_normalize(volume):
    """Z-score normalize non-zero voxels (standard BraTS preprocessing)."""
    out = volume.copy()
    mask = out > 0
    if mask.sum() == 0:
        return out
    mean = out[mask].mean()
    std = out[mask].std()
    if std < 1e-8:
        return out
    out[mask] = (out[mask] - mean) / std
    return out


def load_case_volumes(case):
    modality_vols = {}
    for mod in GRAPH["modalities"]:
        vol = load_volume(case["files"][mod])
        modality_vols[mod] = zscore_normalize(vol)

    gt_seg = load_volume(case["files"]["seg"]).astype(np.int32)
    shape = modality_vols[GRAPH["slic_modality"]].shape

    return modality_vols, gt_seg, shape


# ── Step 1: 3D SLIC Supervoxel Generation ────────────────────────────


def generate_supervoxels(t1n_volume, n_segments=None, compactness=None):
    n_segments = n_segments or GRAPH["n_segments"]
    compactness = compactness or GRAPH["compactness"]

    sv_labels = slic(
        t1n_volume,
        n_segments=n_segments,
        compactness=compactness,
        start_label=0,
        enforce_connectivity=True,
        channel_axis=None,          # input is single-channel 3D
    )

    return sv_labels.astype(np.int32)


# ── Step 2: Dynamic Background Pruning ───────────────────────────────


def prune_background(sv_labels, t1n_volume, min_volume=None):
    min_volume = min_volume or GRAPH["min_sv_volume"]
    unique_labels = np.unique(sv_labels)

    # Compute per-SV statistics
    sv_stats = {}
    for label in unique_labels:
        mask = sv_labels == label
        voxels = t1n_volume[mask]
        sv_stats[label] = {
            "mean_intensity": float(voxels.mean()),
            "volume": int(mask.sum()),
        }

    # Filter by minimum volume first
    volume_ok = {
        l for l, s in sv_stats.items()
        if s["volume"] >= min_volume
    }

    # Sort remaining SVs by mean intensity (ascending)
    sorted_labels = sorted(volume_ok, key=lambda l: sv_stats[l]["mean_intensity"])
    sorted_means = [sv_stats[l]["mean_intensity"] for l in sorted_labels]

    if len(sorted_means) < 2:
        # Edge case: can't compute gaps with fewer than 2 SVs
        return list(volume_ok), {
            "theta": 0.0,
            "total_svs": len(unique_labels),
            "retained": len(volume_ok),
            "pruned_bg": 0,
            "pruned_small": len(unique_labels) - len(volume_ok),
        }

    # Find largest gap in sorted mean distribution
    gaps = np.diff(sorted_means)
    g = int(np.argmax(gaps))
    theta = 0.5 * (sorted_means[g] + sorted_means[g + 1])

    # Retain SVs with mean intensity above threshold
    retained = [l for l in sorted_labels if sv_stats[l]["mean_intensity"] > theta]
    pruned_bg = len(sorted_labels) - len(retained)
    pruned_small = len(unique_labels) - len(volume_ok)

    pruning_info = {
        "theta": float(theta),
        "total_svs": len(unique_labels),
        "after_volume_filter": len(volume_ok),
        "retained": len(retained),
        "pruned_bg": pruned_bg,
        "pruned_small": pruned_small,
        "sv_stats": sv_stats,
    }

    return retained, pruning_info


# ── Step 3: Supervoxel-Level Ground Truth ────────────────────────────


def compute_sv_targets(sv_labels, gt_seg, retained_labels, tau=None):
    tau = tau if tau is not None else GRAPH["tau"]
    num_classes = GRAPH["num_classes"]

    # Pre-compute voxel coordinate arrays for centroid calculation
    coords = np.array(np.where(sv_labels >= 0))  # (3, total_voxels)

    targets = {}
    tumor_count = 0
    healthy_count = 0

    for label in retained_labels:
        mask = sv_labels == label
        gt_voxels = gt_seg[mask].astype(np.int64)
        total = gt_voxels.size

        # ── Tumor proportion (any class > 0 is tumor) ──
        n_tumor = int((gt_voxels > 0).sum())
        y_reg = n_tumor / total if total > 0 else 0.0

        # ── Binary tumor label ──
        y_cls = 1 if y_reg > tau else 0

        # ── Dominant class (mode) ──
        counts = np.bincount(gt_voxels, minlength=num_classes)
        y_dominant = int(counts.argmax())

        # ── Class proportions vector ──
        y_props = (counts / total).astype(np.float32) if total > 0 else np.zeros(num_classes, dtype=np.float32)

        # ── Centroid (mean voxel coordinate) ──
        sv_coords = np.argwhere(mask)  # (N_voxels, 3)
        centroid = sv_coords.mean(axis=0).astype(np.float32)  # (3,)

        targets[label] = {
            "y_reg": float(y_reg),
            "y_cls": int(y_cls),
            "y_dominant": int(y_dominant),
            "y_props": y_props,
            "volume": int(total),
            "centroid": centroid,
            "class_counts": counts.astype(np.int64),
        }

        if y_cls == 1:
            tumor_count += 1
        else:
            healthy_count += 1

    summary = {
        "total_svs": len(retained_labels),
        "tumor_svs": tumor_count,
        "healthy_svs": healthy_count,
        "tumor_ratio": tumor_count / len(retained_labels) if retained_labels else 0.0,
        "mean_tumor_proportion": float(np.mean([t["y_reg"] for t in targets.values()])),
    }

    return targets, summary


# ── Step 4: Segmentation Prior Features ──────────────────────────────


def load_seg_probabilities(case_id, prob_dir=None):
    if prob_dir is None:
        prob_dir = Path("brats_outputs/seg_probs")
    else:
        prob_dir = Path(prob_dir)

    prob_path = prob_dir / f"{case_id}_seg_probs.npy"
    if not prob_path.exists():
        return None

    return np.load(str(prob_path))


def compute_seg_prior_features(sv_labels, retained_labels, seg_probs):
    num_classes = seg_probs.shape[0]
    seg_features = {}

    all_entropies = []

    for label in retained_labels:
        mask = sv_labels == label

        # Mean probability vector across all voxels in this SV
        # seg_probs[:, mask] has shape (num_classes, n_voxels)
        seg_feat = seg_probs[:, mask].mean(axis=1).astype(np.float32)  # (4,)

        # Prediction entropy: -Σ p_c log(p_c)  (clipped to avoid log(0))
        p_clipped = np.clip(seg_feat, 1e-8, 1.0)
        seg_entropy = float(-np.sum(p_clipped * np.log(p_clipped)))

        # Predicted class from mean probabilities
        seg_pred = int(seg_feat.argmax())

        seg_features[label] = {
            "seg_feat": seg_feat,
            "seg_entropy": seg_entropy,
            "seg_pred": seg_pred,
        }

        all_entropies.append(seg_entropy)

    # Summary statistics
    entropies = np.array(all_entropies)
    pred_counts = {}
    for sf in seg_features.values():
        c = sf["seg_pred"]
        pred_counts[c] = pred_counts.get(c, 0) + 1

    seg_summary = {
        "mean_entropy": float(entropies.mean()),
        "std_entropy": float(entropies.std()),
        "max_entropy": float(entropies.max()),
        "high_uncertainty_svs": int((entropies > 1.0).sum()),  # entropy > 1.0 nats
        "pred_class_distribution": pred_counts,
    }

    return seg_features, seg_summary

In [ ]:
# ── Step 5: Patch Extraction ─────────────────────────────────────────


def extract_patches(sv_labels, modality_vols, retained_labels, targets,
                    n_patch=None, patch_neighbors=None):
    n_patch = n_patch or GRAPH["n_patch"]
    s = patch_neighbors or GRAPH["patch_neighbors"]
    mod_names = list(modality_vols.keys())
    n_mods = len(mod_names)

    patches = {}
    padded_count = 0

    for label in retained_labels:
        voxel_coords = np.argwhere(sv_labels == label)  # (N_vox, 3)
        N = voxel_coords.shape[0]

        # k-means++ centroid selection
        actual_k = min(n_patch, N)
        if actual_k < 2:
            centroids = voxel_coords[:actual_k].astype(np.float64)
        else:
            km = KMeans(
                n_clusters=actual_k, init="k-means++",
                n_init=1, max_iter=20, random_state=42,
            )
            km.fit(voxel_coords)
            centroids = km.cluster_centers_  # (actual_k, 3)

        # Build KDTree for nearest-neighbor queries within this SV
        tree = cKDTree(voxel_coords)

        rows = []
        for ci in range(actual_k):
            centroid = centroids[ci]
            k_query = min(s, N)
            _, nn_idx = tree.query(centroid, k=k_query)
            nn_idx = np.atleast_1d(nn_idx)
            nn_coords = voxel_coords[nn_idx]  # (k_query, 3)

            for mod in mod_names:
                vol = modality_vols[mod]
                values = vol[nn_coords[:, 0], nn_coords[:, 1], nn_coords[:, 2]]

                # Pad if fewer neighbors than s
                if len(values) < s:
                    values = np.pad(values, (0, s - len(values)), mode="edge")
                    padded_count += 1

                # Augment with centroid XYZ
                row = np.concatenate([values, centroid])  # (s + 3,)
                rows.append(row)

        # Pad missing centroids if SV was very small
        while len(rows) < n_patch * n_mods:
            rows.append(rows[-1].copy())
            padded_count += 1

        patches[label] = np.stack(rows, axis=0).astype(np.float32)

    patch_info = {
        "n_patch": n_patch,
        "patch_neighbors": s,
        "n_modalities": n_mods,
        "tensor_shape": f"({n_patch * n_mods}, {s + 3})",
        "padded_patches": padded_count,
    }

    return patches, patch_info


# ── Step 6: kNN Graph Construction ───────────────────────────────────


def build_knn_graph(retained_labels, targets, k=None):
    k = k or GRAPH["knn_k"]
    N = len(retained_labels)

    # Map SV labels to consecutive node indices
    label_to_idx = {label: i for i, label in enumerate(retained_labels)}
    idx_to_label = {i: label for label, i in label_to_idx.items()}

    # Collect centroids in index order
    centroids = np.stack(
        [targets[retained_labels[i]]["centroid"] for i in range(N)],
        axis=0,
    )  # (N, 3)

    # kNN via KDTree
    actual_k = min(k + 1, N)  # +1 because query includes self
    tree = cKDTree(centroids)
    dists, indices = tree.query(centroids, k=actual_k)  # (N, actual_k)

    # Build edge list (skip self-loops at index 0)
    src_list = []
    dst_list = []
    attr_list = []

    for i in range(N):
        for j_pos in range(1, actual_k):  # skip self
            j = indices[i, j_pos]
            delta = centroids[j] - centroids[i]  # (3,)
            dist = dists[i, j_pos]

            src_list.append(i)
            dst_list.append(j)
            attr_list.append([delta[0], delta[1], delta[2], dist])

    # Symmetrize: add reverse edges (deduplicated)
    edge_set = set()
    sym_src, sym_dst, sym_attr = [], [], []
    for idx in range(len(src_list)):
        s, d = src_list[idx], dst_list[idx]
        if (s, d) not in edge_set:
            edge_set.add((s, d))
            sym_src.append(s)
            sym_dst.append(d)
            sym_attr.append(attr_list[idx])
        if (d, s) not in edge_set:
            edge_set.add((d, s))
            rev_delta = [-attr_list[idx][0], -attr_list[idx][1],
                         -attr_list[idx][2], attr_list[idx][3]]
            sym_src.append(d)
            sym_dst.append(s)
            sym_attr.append(rev_delta)

    if len(sym_src) > 0:
        edge_index = np.array([sym_src, sym_dst], dtype=np.int64)   # (2, E)
        edge_attr = np.array(sym_attr, dtype=np.float32).reshape(-1, 4)  # (E, 4)
    else:
        edge_index = np.zeros((2, 0), dtype=np.int64)
        edge_attr = np.zeros((0, 4), dtype=np.float32)

    # Degree statistics
    degrees = np.bincount(edge_index[0] if edge_index.shape[1] > 0 else [], minlength=N)

    graph_info = {
        "num_nodes": N,
        "num_edges": edge_index.shape[1],
        "k": k,
        "mean_degree": float(degrees.mean()),
        "min_degree": int(degrees.min()) if len(degrees) > 0 else 0,
        "max_degree": int(degrees.max()) if len(degrees) > 0 else 0,
        "avg_edge_dist": float(edge_attr[:, 3].mean()) if edge_attr.shape[0] > 0 else 0.0,
    }

    return edge_index, edge_attr, label_to_idx, idx_to_label, graph_info


# ── Step 7: Edge-Level Ground Truth ──────────────────────────────────

# Symmetric boundary type encoding: ordered pair (min, max) of class IDs
# 10 types for 4 classes: (0,0),(0,1),(0,2),(0,3),(1,1),(1,2),(1,3),(2,2),(2,3),(3,3)
BOUNDARY_TYPE_MAP = {}
_bt_idx = 0
for _a in range(4):
    for _b in range(_a, 4):
        BOUNDARY_TYPE_MAP[(_a, _b)] = _bt_idx
        _bt_idx += 1
BOUNDARY_TYPE_NAMES = {
    v: f"{GRAPH['class_names'][a]}↔{GRAPH['class_names'][b]}"
    for (a, b), v in BOUNDARY_TYPE_MAP.items()
}


def compute_edge_targets(edge_index, targets, idx_to_label):
    E = edge_index.shape[1]
    y_binary = np.zeros(E, dtype=np.int64)
    y_type = np.zeros(E, dtype=np.int64)
    y_grad = np.zeros(E, dtype=np.float32)

    for e in range(E):
        i_idx, j_idx = int(edge_index[0, e]), int(edge_index[1, e])
        label_i, label_j = idx_to_label[i_idx], idx_to_label[j_idx]

        dom_i = targets[label_i]["y_dominant"]
        dom_j = targets[label_j]["y_dominant"]
        reg_i = targets[label_i]["y_reg"]
        reg_j = targets[label_j]["y_reg"]

        # Binary: same class?
        y_binary[e] = 1 if dom_i == dom_j else 0

        # Boundary type: symmetric ordered pair
        pair = (min(dom_i, dom_j), max(dom_i, dom_j))
        y_type[e] = BOUNDARY_TYPE_MAP[pair]

        # Transition gradient
        y_grad[e] = abs(reg_i - reg_j)

    # Statistics
    type_counts = {}
    for t in y_type:
        name = BOUNDARY_TYPE_NAMES[int(t)]
        type_counts[name] = type_counts.get(name, 0) + 1

    edge_target_info = {
        "num_edges": E,
        "same_class_edges": int(y_binary.sum()),
        "diff_class_edges": int((1 - y_binary).sum()),
        "boundary_type_distribution": type_counts,
        "mean_gradient": float(y_grad.mean()) if E > 0 else 0.0,
        "max_gradient": float(y_grad.max()) if E > 0 else 0.0,
    }

    edge_targets = {
        "y_edge_binary": y_binary,
        "y_edge_type": y_type,
        "y_edge_grad": y_grad,
    }

    return edge_targets, edge_target_info


# ── Full Pipeline (Steps 1–7) ────────────────────────────────────────


def preprocess_case(case, config=None, seg_prob_dir=None):
    cfg = config or GRAPH
    case_id = case["case_id"]

    # Load native-resolution volumes
    modality_vols, gt_seg, shape = load_case_volumes(case)
    t1n = modality_vols[cfg["slic_modality"]]

    # Step 1: 3D SLIC
    sv_labels = generate_supervoxels(
        t1n, n_segments=cfg["n_segments"], compactness=cfg["compactness"]
    )

    # Step 2: Dynamic background pruning
    retained_labels, pruning_info = prune_background(
        sv_labels, t1n, min_volume=cfg["min_sv_volume"]
    )

    # Step 3: Ground truth
    targets, target_summary = compute_sv_targets(
        sv_labels, gt_seg, retained_labels, tau=cfg["tau"]
    )

    # Step 4: Segmentation prior features (optional)
    seg_features = None
    seg_summary = None
    if seg_prob_dir is not None:
        seg_probs = load_seg_probabilities(case_id, seg_prob_dir)
        if seg_probs is not None:
            seg_features, seg_summary = compute_seg_prior_features(
                sv_labels, retained_labels, seg_probs
            )
            del seg_probs

    # Step 5: Patch extraction
    patches, patch_info = extract_patches(
        sv_labels, modality_vols, retained_labels, targets,
        n_patch=cfg["n_patch"], patch_neighbors=cfg["patch_neighbors"],
    )

    # Step 6: kNN graph construction
    edge_index, edge_attr, label_to_idx, idx_to_label, graph_info = (
        build_knn_graph(retained_labels, targets, k=cfg["knn_k"])
    )

    # Step 7: Edge-level ground truth
    edge_targets, edge_target_info = compute_edge_targets(
        edge_index, targets, idx_to_label
    )

    return {
        "case_id": case_id,
        "sv_labels": sv_labels,
        "retained_labels": retained_labels,
        "targets": targets,
        "pruning_info": pruning_info,
        "target_summary": target_summary,
        "seg_features": seg_features,
        "seg_summary": seg_summary,
        "patches": patches,
        "patch_info": patch_info,
        "edge_index": edge_index,
        "edge_attr": edge_attr,
        "edge_targets": edge_targets,
        "edge_target_info": edge_target_info,
        "label_to_idx": label_to_idx,
        "idx_to_label": idx_to_label,
        "graph_info": graph_info,
        "modality_vols": modality_vols,
        "gt_seg": gt_seg,
        "shape": shape,
    }

### Test Preprocessing

In [ ]:
cases = discover_cases(GRAPH["data_root"])
print(f"Discovered {len(cases)} valid BraTS cases")
if cases:
    t0 = time.time()
    result = preprocess_case(cases[0], seg_prob_dir=str(seg_prob_dir))
    print(f"Case: {result['case_id']} | {len(result['retained_labels'])} nodes, "
          f"{result['edge_index'].shape[1]} edges | {time.time()-t0:.1f}s")

## Shared Backbone (from Plan 1)PatchEmbedder (Transformer) + GraphEncoder (GATv2) — identical to Plan 1.

In [ ]:
# ── Patch-Level Transformer Embedder ─────────────────────────────────


class PatchEmbedder(nn.Module):
    def __init__(self, patch_dim=None, n_modalities=4, n_patch=None,
                 embed_dim=None, n_layers=None, n_heads=None,
                 seg_prior_dim=5, dropout=0.1):
        super().__init__()
        patch_dim = patch_dim or (GRAPH["patch_neighbors"] + 3)
        n_patch = n_patch or GRAPH["n_patch"]
        embed_dim = embed_dim or GRAPH["embed_dim"]
        n_layers = n_layers or GRAPH["transformer_layers"]
        n_heads = n_heads or GRAPH["transformer_heads"]

        self.embed_dim = embed_dim
        self.n_modalities = n_modalities
        self.n_patch = n_patch
        self.n_rows = n_patch * n_modalities  # 16 rows per SV

        # Linear projection: patch row → embed_dim
        self.input_proj = nn.Linear(patch_dim, embed_dim)

        # Learnable modality embeddings (shared across patches of same modality)
        self.modality_embed = nn.Embedding(n_modalities, embed_dim)

        # Learnable patch position embeddings
        self.patch_pos_embed = nn.Embedding(n_patch, embed_dim)

        # [CLS] token
        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)

        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=n_heads,
            dim_feedforward=embed_dim * 4, dropout=dropout,
            batch_first=True, activation="gelu",
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer, num_layers=n_layers,
        )

        # Seg prior projection (optional, 0-dim if no seg priors)
        self.has_seg_prior = seg_prior_dim > 0
        if self.has_seg_prior:
            self.seg_proj = nn.Sequential(
                nn.Linear(seg_prior_dim, embed_dim),
                nn.GELU(),
            )
            # Final MLP: concat([CLS], seg_embed) → embed_dim
            self.out_proj = nn.Sequential(
                nn.Linear(embed_dim * 2, embed_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(embed_dim, embed_dim),
            )
        else:
            self.out_proj = nn.Sequential(
                nn.Linear(embed_dim, embed_dim),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(embed_dim, embed_dim),
            )

    def forward(self, patch_tensors, seg_priors=None, return_attention=False):
        B = patch_tensors.size(0)
        device = patch_tensors.device

        # Project patch rows
        x = self.input_proj(patch_tensors)  # (B, n_rows, embed_dim)

        # Add modality embeddings: rows are ordered as
        # [patch0_mod0, patch0_mod1, ..., patch0_modM, patch1_mod0, ...]
        mod_ids = torch.arange(self.n_modalities, device=device)
        mod_ids = mod_ids.repeat(self.n_patch)  # (n_rows,)
        x = x + self.modality_embed(mod_ids).unsqueeze(0)  # broadcast over B

        # Add patch position embeddings
        patch_ids = torch.arange(self.n_patch, device=device)
        patch_ids = patch_ids.repeat_interleave(self.n_modalities)  # (n_rows,)
        x = x + self.patch_pos_embed(patch_ids).unsqueeze(0)

        # Prepend [CLS] token
        cls = self.cls_token.expand(B, -1, -1)  # (B, 1, embed_dim)
        x = torch.cat([cls, x], dim=1)  # (B, 1+n_rows, embed_dim)

        # Capture attention weights via manual layer-by-layer forward
        patch_attentions = None
        if return_attention:
            patch_attentions = []
            for layer in self.transformer.layers:
                # Manual forward through TransformerEncoderLayer
                # to capture self-attention weights
                x2, attn_w = layer.self_attn(
                    layer.norm1(x), layer.norm1(x), layer.norm1(x),
                    need_weights=True, average_attn_weights=False,
                )
                patch_attentions.append(attn_w.detach())
                # Complete the layer forward
                x = x + layer.dropout1(x2)
                x = x + layer._ff_block(layer.norm2(x))
        else:
            # Standard forward
            x = self.transformer(x)  # (B, 1+n_rows, embed_dim)

        cls_out = x[:, 0]  # (B, embed_dim)

        # Combine with seg prior if available
        if self.has_seg_prior and seg_priors is not None:
            seg_embed = self.seg_proj(seg_priors)  # (B, embed_dim)
            combined = torch.cat([cls_out, seg_embed], dim=-1)
            node_embeds = self.out_proj(combined)
        else:
            node_embeds = self.out_proj(cls_out)

        return node_embeds, patch_attentions

In [ ]:
# ── Graph-Level GATv2 Encoder ────────────────────────────────────────


class GraphEncoder(nn.Module):
    def __init__(self, embed_dim=None, n_layers=None, n_heads=None,
                 edge_dim=4, pe_dim=None, dropout=0.1):
        super().__init__()
        from torch_geometric.nn import GATv2Conv, LayerNorm

        embed_dim = embed_dim or GRAPH["embed_dim"]
        n_layers = n_layers or GRAPH["gat_layers"]
        n_heads = n_heads or GRAPH["gat_heads"]
        pe_dim = pe_dim or GRAPH["laplacian_pe_dim"]

        self.embed_dim = embed_dim
        self.pe_dim = pe_dim

        # Laplacian PE projection
        self.pe_proj = nn.Linear(pe_dim, embed_dim)

        # GATv2 layers with residual connections
        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()
        for _ in range(n_layers):
            self.convs.append(GATv2Conv(
                embed_dim, embed_dim // n_heads, heads=n_heads,
                edge_dim=edge_dim, dropout=dropout, concat=True,
            ))
            self.norms.append(LayerNorm(embed_dim))

        # Multiscale fusion: concat all layer outputs → MLP → embed_dim
        self.fusion = nn.Sequential(
            nn.Linear(embed_dim * n_layers, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, embed_dim),
        )

        self.n_layers = n_layers
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr=None, lap_pe=None,
                return_attention=False):
        # Add Laplacian PE
        if lap_pe is not None:
            x = x + self.pe_proj(lap_pe)

        layer_outputs = []
        alphas = []

        for i, (conv, norm) in enumerate(zip(self.convs, self.norms)):
            residual = x
            if return_attention:
                x, (_, alpha) = conv(
                    x, edge_index, edge_attr=edge_attr,
                    return_attention_weights=True,
                )
                alphas.append(alpha)
            else:
                x = conv(x, edge_index, edge_attr=edge_attr)
            x = norm(x)
            x = x + residual  # residual connection
            if i < self.n_layers - 1:
                x = F.elu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
            layer_outputs.append(x)

        # Multiscale fusion
        fused = torch.cat(layer_outputs, dim=-1)  # (N, embed_dim * n_layers)
        out = self.fusion(fused)  # (N, embed_dim)

        return out, alphas

In [ ]:
# ── Task 2: Edge Boundary Type Classification Head ──────────────────


class EdgeHead(nn.Module):
    def __init__(self, embed_dim=None, edge_dim=4, n_boundary_types=10,
                 dropout=0.2):
        super().__init__()
        embed_dim = embed_dim or GRAPH["embed_dim"]
        in_dim = embed_dim * 3 + edge_dim  # h_i + h_j + |h_i-h_j| + e_ij

        self.mlp = nn.Sequential(
            nn.Linear(in_dim, embed_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(embed_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, n_boundary_types),
        )

    def forward(self, node_feats, edge_index, edge_attr):
        h_i = node_feats[edge_index[0]]       # (E, embed_dim)
        h_j = node_feats[edge_index[1]]       # (E, embed_dim)
        h_diff = torch.abs(h_i - h_j)         # (E, embed_dim)

        x = torch.cat([h_i, h_j, h_diff, edge_attr], dim=-1)  # (E, 3*D+4)
        return self.mlp(x)  # (E, 10)


# ── Laplacian Positional Encoding ────────────────────────────────────


def compute_laplacian_pe(edge_index, num_nodes, k=None):
    from scipy.sparse import coo_matrix
    from scipy.sparse.linalg import eigsh

    k = k or GRAPH["laplacian_pe_dim"]

    if isinstance(edge_index, torch.Tensor):
        edge_index = edge_index.cpu().numpy()

    row = edge_index[0]
    col = edge_index[1]
    data = np.ones(len(row), dtype=np.float64)

    # Build adjacency matrix
    A = coo_matrix((data, (row, col)), shape=(num_nodes, num_nodes)).tocsr()

    # Degree matrix
    deg = np.array(A.sum(axis=1)).flatten()
    deg_sqrt = np.sqrt(np.maximum(deg, 1e-12))
    deg_inv_sqrt = np.where(deg > 0, 1.0 / deg_sqrt, 0.0)

    # Normalized Laplacian: I - D^{-1/2} A D^{-1/2}
    from scipy.sparse import diags
    D_inv_sqrt = diags(deg_inv_sqrt)
    L = diags(np.ones(num_nodes)) - D_inv_sqrt @ A @ D_inv_sqrt

    # Compute k+1 smallest eigenvectors (skip trivial eigenvector 0)
    num_eig = min(k + 1, num_nodes - 1)
    if num_eig < 2:
        return torch.zeros(num_nodes, k, dtype=torch.float32)

    try:
        eigenvalues, eigenvectors = eigsh(L, k=num_eig, which="SM")
        # Skip first eigenvector (constant, eigenvalue ≈ 0)
        pe = eigenvectors[:, 1:k+1]

        # Pad if fewer eigenvectors than k
        if pe.shape[1] < k:
            pad = np.zeros((num_nodes, k - pe.shape[1]))
            pe = np.hstack([pe, pad])

        # Random sign flip for invariance
        signs = np.sign(pe[0])
        signs[signs == 0] = 1
        pe = pe * signs

    except Exception:
        pe = np.zeros((num_nodes, k))

    return torch.from_numpy(np.nan_to_num(pe, nan=0.0)).float()

## Plan 2 Task HeadsThree task-specific heads on top of the shared backbone:1. **RegressionHead:** Ensemble of 4 parallel MLPs with attention pooling → y_reg ∈ [0,1]2. **EdgeHead:** Same as Plan 1 (10-class boundary type)3. **UncertaintyHead:** Binary MLP predicting seg model errors

In [ ]:
# ── Task 1: Tumor Proportion Regression Head ─────────────────────────


class RegressionHead(nn.Module):
    def __init__(self, embed_dim=None, K=None, dropout=0.2):
        super().__init__()
        embed_dim = embed_dim or GRAPH["embed_dim"]
        K = K or GRAPH["regression_ensemble_size"]

        self.K = K

        # K parallel MLP heads
        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(embed_dim, 64),
                nn.GELU(),
                nn.Dropout(dropout),
                nn.Linear(64, 1),
            )
            for _ in range(K)
        ])

        # Attention gate: learns which ensemble members to trust
        self.attention = nn.Sequential(
            nn.Linear(embed_dim, K),
            nn.Softmax(dim=-1),
        )

    def forward(self, x):
       # Each head produces a raw prediction
        preds = torch.stack([head(x).squeeze(-1) for head in self.heads],
                            dim=-1)  # (N, K)

        # Attention-weighted combination
        attn_weights = self.attention(x)  # (N, K)
        combined = (preds * attn_weights).sum(dim=-1)  # (N,)

        # Sigmoid to clamp to [0, 1]
        y_reg = torch.sigmoid(combined)

        # Also return individual sigmoid predictions for uncertainty
        ensemble_preds = torch.sigmoid(preds)

        return y_reg, ensemble_preds

In [ ]:
# ── Task 3: Segmentation Uncertainty Prediction Head ─────────────────


class UncertaintyHead(nn.Module):
    def __init__(self, embed_dim=None, dropout=0.2):
        super().__init__()
        embed_dim = embed_dim or GRAPH["embed_dim"]
        self.mlp = nn.Sequential(
            nn.Linear(embed_dim, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, x):
        return self.mlp(x).squeeze(-1)


# ── Uncertainty-Weighted Multi-Task Loss ─────────────────────────────


class MultiTaskLoss(nn.Module):
    def __init__(self, init_log_vars=None):
        super().__init__()
        init_log_vars = init_log_vars or [
            GRAPH["init_log_var_reg"],
            GRAPH["init_log_var_edge"],
            GRAPH["init_log_var_unc"],
        ]
        # Learnable parameters: log(σ²) for each task
        self.log_vars = nn.ParameterList([
            nn.Parameter(torch.tensor(v, dtype=torch.float32))
            for v in init_log_vars
        ])

    def forward(self, losses):
        total = torch.tensor(0.0, device=losses[0].device)
        weights = []

        for i, (loss, log_var) in enumerate(zip(losses, self.log_vars)):
            precision = torch.exp(-log_var)  # 1/σ²
            total = total + 0.5 * precision * loss + 0.5 * log_var
            weights.append(float(precision.item()))

        return total, weights

    def get_sigmas(self):
        return [float(torch.exp(0.5 * lv).item()) for lv in self.log_vars]

### MultiTaskRefinerFull model combining shared backbone + 3 task heads + uncertainty-weighted loss.

In [ ]:
# ── Full Model ───────────────────────────────────────────────────────


class MultiTaskRefiner(nn.Module):
    def __init__(self, config=None, use_seg_prior=True):
        super().__init__()
        cfg = config or GRAPH
        seg_dim = 5 if use_seg_prior else 0

        # Shared backbone (same architecture as Plan 1)
        self.embedder = PatchEmbedder(
            patch_dim=cfg["patch_neighbors"] + 3,
            n_modalities=len(cfg["modalities"]),
            n_patch=cfg["n_patch"],
            embed_dim=cfg["embed_dim"],
            n_layers=cfg["transformer_layers"],
            n_heads=cfg["transformer_heads"],
            seg_prior_dim=seg_dim,
        )
        self.encoder = GraphEncoder(
            embed_dim=cfg["embed_dim"],
            n_layers=cfg["gat_layers"],
            n_heads=cfg["gat_heads"],
            edge_dim=4,
            pe_dim=cfg["laplacian_pe_dim"],
        )

        # Task heads
        self.regression_head = RegressionHead(
            embed_dim=cfg["embed_dim"],
            K=cfg["regression_ensemble_size"],
        )
        self.edge_head = EdgeHead(
            embed_dim=cfg["embed_dim"],
            edge_dim=4,
            n_boundary_types=cfg["n_boundary_types"],
        )
        self.uncertainty_head = UncertaintyHead(embed_dim=cfg["embed_dim"])

        # Multi-task loss
        self.loss_fn = MultiTaskLoss()

        self.use_seg_prior = use_seg_prior

    def forward(self, patch_tensors, seg_priors, edge_index, edge_attr,
                lap_pe=None, return_attention=False):
        # Stage 1: Patch-level embedding
        sp = seg_priors if self.use_seg_prior else None
        node_feats, patch_attns = self.embedder(
            patch_tensors, sp, return_attention=return_attention,
        )

        # Stage 2: Graph-level encoding
        graph_feats, graph_attns = self.encoder(
            node_feats, edge_index, edge_attr=edge_attr,
            lap_pe=lap_pe, return_attention=return_attention,
        )

        # Task 1: Tumor proportion regression
        y_reg, ensemble_preds = self.regression_head(graph_feats)

        # Task 2: Edge boundary classification
        edge_logits = self.edge_head(graph_feats, edge_index, edge_attr)

        # Task 3: Segmentation uncertainty prediction
        unc_logits = self.uncertainty_head(graph_feats)

        outputs = {
            "y_reg": y_reg,
            "ensemble_preds": ensemble_preds,
            "edge_logits": edge_logits,
            "unc_logits": unc_logits,
        }

        attention_dict = {
            "graph": graph_attns,
            "patch": patch_attns,
        }

        return outputs, attention_dict

    def compute_loss(self, outputs, targets_dict):
        # Task 1: Smooth L1 (Huber) loss for regression
        L_reg = F.smooth_l1_loss(
            outputs["y_reg"], targets_dict["y_reg_target"],
        )

        # Task 2: Cross-entropy for edge classification
        edge_weight = targets_dict.get("edge_class_weights", None)
        L_edge = F.cross_entropy(
            outputs["edge_logits"], targets_dict["edge_type_target"],
            weight=edge_weight,
        )

        # Task 3: BCE for uncertainty prediction
        L_unc = F.binary_cross_entropy_with_logits(
            outputs["unc_logits"], targets_dict["unc_target"],
        )

        # Combine with learned uncertainty weights
        total, weights = self.loss_fn([L_reg, L_edge, L_unc])

        sigmas = self.loss_fn.get_sigmas()
        loss_dict = {
            "total": total.item(),
            "L_reg": L_reg.item(),
            "L_edge": L_edge.item(),
            "L_unc": L_unc.item(),
            "w_reg": weights[0],
            "w_edge": weights[1],
            "w_unc": weights[2],
            "σ_reg": sigmas[0],
            "σ_edge": sigmas[1],
            "σ_unc": sigmas[2],
        }

        return total, loss_dict

    def count_parameters(self):
        """Count total and per-component trainable parameters."""
        components = {
            "PatchEmbedder": self.embedder,
            "GraphEncoder": self.encoder,
            "RegressionHead": self.regression_head,
            "EdgeHead": self.edge_head,
            "UncertaintyHead": self.uncertainty_head,
            "MultiTaskLoss": self.loss_fn,
        }
        total = 0
        breakdown = {}
        for name, module in components.items():
            n = sum(p.numel() for p in module.parameters() if p.requires_grad)
            breakdown[name] = n
            total += n
        breakdown["Total"] = total
        return breakdown

### Sanity Check — Synthetic Forward + Loss + Backward

In [ ]:
model = MultiTaskRefiner(use_seg_prior=True).to(DEVICE)
params = model.count_parameters()
print("Parameter count:")
for name, n in params.items():
    print(f"  {name}: {n:,}")

# Synthetic data
N, E = 100, 400
patch_tensors = torch.randn(N, 16, 19, device=DEVICE)
seg_priors = torch.rand(N, 5, device=DEVICE)
edge_index = torch.randint(0, N, (2, E), device=DEVICE)
edge_attr = torch.randn(E, 4, device=DEVICE)
lap_pe = torch.randn(N, 8, device=DEVICE)

# Forward
model.eval()
with torch.no_grad():
    outputs, attn = model(patch_tensors, seg_priors, edge_index, edge_attr, lap_pe=lap_pe, return_attention=True)

print(f"\nTask 1: y_reg {outputs['y_reg'].shape}, range=[{outputs['y_reg'].min():.4f}, {outputs['y_reg'].max():.4f}]")
print(f"  Ensemble: {outputs['ensemble_preds'].shape}, var={outputs['ensemble_preds'].var(-1).mean():.6f}")
print(f"Task 2: edge_logits {outputs['edge_logits'].shape}")
unc_p = torch.sigmoid(outputs['unc_logits'])
print(f"Task 3: unc prob range=[{unc_p.min():.4f}, {unc_p.max():.4f}]")

# Loss + backward
model.train()
out_train, _ = model(patch_tensors, seg_priors, edge_index, edge_attr, lap_pe=lap_pe)
targets = {
    "y_reg_target": torch.rand(N, device=DEVICE),
    "edge_type_target": torch.randint(0, 10, (E,), device=DEVICE),
    "unc_target": torch.randint(0, 2, (N,), device=DEVICE).float(),
}
total_loss, loss_dict = model.compute_loss(out_train, targets)
total_loss.backward()

print(f"\nLoss: total={loss_dict['total']:.4f}")
print(f"  L_reg={loss_dict['L_reg']:.4f} (σ={loss_dict['σ_reg']:.3f})")
print(f"  L_edge={loss_dict['L_edge']:.4f} (σ={loss_dict['σ_edge']:.3f})")
print(f"  L_unc={loss_dict['L_unc']:.4f} (σ={loss_dict['σ_unc']:.3f})")
print(f"Model memory: {sum(p.numel() * p.element_size() for p in model.parameters()) / 1e6:.1f} MB")

### Real Data Test

In [ ]:
if cases:
    result = preprocess_case(cases[0], seg_prob_dir=str(seg_prob_dir))
    retained = result["retained_labels"]
    N_real = len(retained)

    patch_batch = torch.stack([torch.from_numpy(result["patches"][l]) for l in retained]).to(DEVICE)
    seg_batch = torch.zeros(N_real, 5, device=DEVICE)
    if result["seg_features"] is not None:
        seg_list = [torch.from_numpy(np.concatenate([result["seg_features"][l]["seg_feat"],
                    [result["seg_features"][l]["seg_entropy"]]])) for l in retained]
        seg_batch = torch.stack(seg_list).float().to(DEVICE)

    ei = torch.from_numpy(result["edge_index"]).to(DEVICE)
    ea = torch.from_numpy(result["edge_attr"]).to(DEVICE)
    lpe = compute_laplacian_pe(result["edge_index"], N_real).to(DEVICE)

    model.eval()
    with torch.no_grad():
        out, _ = model(patch_batch, seg_batch, ei, ea, lap_pe=lpe)

    gt_reg = torch.tensor([result["targets"][l]["y_reg"] for l in retained], device=DEVICE)
    mae = (out["y_reg"] - gt_reg).abs().mean()
    gt_edge = torch.from_numpy(result["edge_targets"]["y_edge_type"]).to(DEVICE)
    edge_acc = (out["edge_logits"].argmax(-1) == gt_edge).float().mean()

    print(f"Case: {result['case_id']} | Nodes: {N_real}, Edges: {ei.shape[1]}")
    print(f"Task 1: MAE={mae:.4f}, y_reg range=[{out['y_reg'].min():.4f}, {out['y_reg'].max():.4f}]")
    print(f"Task 2: Edge acc={edge_acc:.1%}")
    print(f"Task 3: Unc prob range=[{torch.sigmoid(out['unc_logits']).min():.4f}, {torch.sigmoid(out['unc_logits']).max():.4f}]")
    print("End-to-end verified ✓")

## TrainingMulti-task training with Kendall uncertainty-weighted loss. AdamW + cosine annealing.

In [ ]:
def assemble_tensors(result, device):
    retained = result["retained_labels"]
    N = len(retained)
    patch_batch = torch.stack([torch.from_numpy(result["patches"][l]) for l in retained]).to(device)

    if result["seg_features"] is not None:
        seg_list = [torch.from_numpy(np.concatenate([result["seg_features"][l]["seg_feat"],
                    [result["seg_features"][l]["seg_entropy"]]])) for l in retained]
        seg_batch = torch.stack(seg_list).float().to(device)
    else:
        seg_batch = torch.zeros(N, 5, device=device)

    ei = torch.from_numpy(result["edge_index"]).to(device)
    ea = torch.from_numpy(result["edge_attr"]).to(device)
    lpe = compute_laplacian_pe(result["edge_index"], N).to(device)

    gt_reg = torch.tensor([result["targets"][l]["y_reg"] for l in retained], dtype=torch.float32, device=device)
    gt_edge = torch.from_numpy(result["edge_targets"]["y_edge_type"]).to(device)

    # Uncertainty GT: was seg model wrong? (1 if seg_pred != y_dominant)
    if result["seg_features"] is not None:
        gt_unc = torch.tensor([
            1.0 if result["seg_features"][l]["seg_pred"] != result["targets"][l]["y_dominant"] else 0.0
            for l in retained], dtype=torch.float32, device=device)
    else:
        gt_unc = torch.zeros(N, dtype=torch.float32, device=device)

    targets = {"y_reg_target": gt_reg, "edge_type_target": gt_edge, "unc_target": gt_unc}
    return patch_batch, seg_batch, ei, ea, lpe, targets


def train_one_epoch(model, graphs, optimizer, scheduler, accum_steps=4):
    model.train()
    total_loss = 0.0
    metrics = {"L_reg": 0, "L_edge": 0, "L_unc": 0, "n_node": 0, "n_edge": 0,
               "node_mae": 0, "edge_correct": 0}
    optimizer.zero_grad()

    processed = 0
    for i, result in enumerate(graphs):
        if result["edge_index"].shape[1] == 0:
            continue
        patches, seg, ei, ea, lpe, targets = assemble_tensors(result, DEVICE)
        outputs, _ = model(patches, seg, ei, ea, lap_pe=lpe)
        loss, ld = model.compute_loss(outputs, targets)
        if torch.isnan(loss) or torch.isinf(loss):
            continue
        (loss / accum_steps).backward()
        processed += 1

        if (i + 1) % accum_steps == 0 or (i + 1) == len(graphs):
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        total_loss += loss.item()
        metrics["L_reg"] += ld["L_reg"]
        metrics["L_edge"] += ld["L_edge"]
        metrics["L_unc"] += ld["L_unc"]
        metrics["node_mae"] += (outputs["y_reg"] - targets["y_reg_target"]).abs().sum().item()
        metrics["n_node"] += len(targets["y_reg_target"])
        metrics["edge_correct"] += (outputs["edge_logits"].argmax(-1) == targets["edge_type_target"]).sum().item()
        metrics["n_edge"] += len(targets["edge_type_target"])

    n = max(processed, 1)
    return {
        "loss": total_loss / n,
        "L_reg": metrics["L_reg"] / n, "L_edge": metrics["L_edge"] / n, "L_unc": metrics["L_unc"] / n,
        "mae": metrics["node_mae"] / max(metrics["n_node"], 1),
        "edge_acc": metrics["edge_correct"] / max(metrics["n_edge"], 1),
        "sigmas": model.loss_fn.get_sigmas(),
    }


@torch.no_grad()
def evaluate(model, graphs):
    model.eval()
    total_loss = 0.0
    metrics = {"node_mae": 0, "n_node": 0, "edge_correct": 0, "n_edge": 0}

    for result in graphs:
        if result["edge_index"].shape[1] == 0:
            continue
        patches, seg, ei, ea, lpe, targets = assemble_tensors(result, DEVICE)
        outputs, _ = model(patches, seg, ei, ea, lap_pe=lpe)
        loss, _ = model.compute_loss(outputs, targets)
        total_loss += loss.item()
        metrics["node_mae"] += (outputs["y_reg"] - targets["y_reg_target"]).abs().sum().item()
        metrics["n_node"] += len(targets["y_reg_target"])
        metrics["edge_correct"] += (outputs["edge_logits"].argmax(-1) == targets["edge_type_target"]).sum().item()
        metrics["n_edge"] += len(targets["edge_type_target"])

    n = max(processed, 1)
    return {
        "loss": total_loss / n,
        "mae": metrics["node_mae"] / max(metrics["n_node"], 1),
        "edge_acc": metrics["edge_correct"] / max(metrics["n_edge"], 1),
    }

print("Training functions defined ✓")

In [ ]:
MAX_CASES = 50
cases_to_use = cases[:MAX_CASES]

print(f"Preprocessing {len(cases_to_use)} cases...")
all_graphs = []
for i, case in enumerate(cases_to_use):
    t0 = time.time()
    result = preprocess_case(case, seg_prob_dir=str(seg_prob_dir))
    if result["edge_index"].shape[1] < 10:
        print(f"  [{i+1}/{len(cases_to_use)}] {case['case_id']}: SKIPPED")
        continue
    all_graphs.append(result)
    seg_str = "✓ seg" if result["seg_features"] is not None else "no seg"
    print(f"  [{i+1}/{len(cases_to_use)}] {case['case_id']}: {len(result['retained_labels'])} nodes, {seg_str} ({time.time()-t0:.1f}s)")

print(f"\nUsable: {len(all_graphs)}")
split = int(0.8 * len(all_graphs))
train_graphs, val_graphs = all_graphs[:split], all_graphs[split:]
print(f"Train: {len(train_graphs)}, Val: {len(val_graphs)}")

# Edge class weights
edge_counts = np.zeros(10, dtype=np.float64)
for g in train_graphs:
    for t in g["edge_targets"]["y_edge_type"]: edge_counts[int(t)] += 1
edge_counts = np.maximum(edge_counts, 1.0)
ew = 1.0 / edge_counts; ew = ew / ew.sum() * 10
edge_class_weights = torch.tensor(ew, dtype=torch.float32, device=DEVICE)

model = MultiTaskRefiner(use_seg_prior=True).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=GRAPH["lr"], weight_decay=GRAPH["weight_decay"])
total_steps = GRAPH["epochs"] * len(train_graphs) // GRAPH["accum_steps"]
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(total_steps, 1))

best_val_loss, patience, pat_ctr = float("inf"), 15, 0
Path("checkpoints").mkdir(exist_ok=True)

print(f"\nTraining for up to {GRAPH['epochs']} epochs...")
for epoch in range(1, GRAPH["epochs"] + 1):
    random.shuffle(train_graphs)
    tm = train_one_epoch(model, train_graphs, optimizer, scheduler, GRAPH["accum_steps"])
    if epoch % GRAPH["eval_every"] == 0 or epoch == 1:
        vm = evaluate(model, val_graphs)
        sig = tm["sigmas"]
        print(f"Ep {epoch:3d} | Train loss={tm['loss']:.4f} mae={tm['mae']:.4f} edge={tm['edge_acc']:.1%} | "
              f"Val loss={vm['loss']:.4f} mae={vm['mae']:.4f} edge={vm['edge_acc']:.1%} | "
              f"σ=[{sig[0]:.3f},{sig[1]:.3f},{sig[2]:.3f}]")
        if vm["loss"] < best_val_loss:
            best_val_loss = vm["loss"]; pat_ctr = 0
            torch.save(model.state_dict(), str(GRAPH["checkpoint"]))
        else:
            pat_ctr += GRAPH["eval_every"]
            if pat_ctr >= patience:
                print(f"\nEarly stopping at epoch {epoch}"); break

if GRAPH["checkpoint"].exists():
    model.load_state_dict(torch.load(str(GRAPH["checkpoint"]), map_location=DEVICE, weights_only=True))
    print(f"\nLoaded best (val loss={best_val_loss:.4f})")

## Explainability (5 Levels)1. **Graph Attention:** GATv2 neighbor influence ranking2. **Patch Attention:** Transformer [CLS]→patch modality importance3. **Regression Refinement:** MAE, R², zero-shot Dice, ensemble variance4. **Task Divergence:** Cross-task agreement/disagreement categorization5. **Uncertainty-Driven:** Why does the model predict seg error? Does regression compensate?

In [ ]:
# ── Graph-Level Attention Traces ─────────────────────────────────────


def graph_attention_trace(node_idx, edge_index, graph_attentions,
                          targets=None, idx_to_label=None, top_k=5):
    if isinstance(edge_index, np.ndarray):
        edge_index = torch.from_numpy(edge_index)

    # Find all edges where node_idx is the destination
    dst_mask = (edge_index[1] == node_idx)
    incoming_indices = torch.where(dst_mask)[0]

    if len(incoming_indices) == 0:
        return {
            "node_idx": node_idx,
            "sv_label": idx_to_label.get(node_idx) if idx_to_label else None,
            "neighbors": [],
            "attention_entropy": 0.0,
            "total_incoming_edges": 0,
        }

    # Source nodes for incoming edges
    src_nodes = edge_index[0, incoming_indices].cpu().numpy()
    unique_src = np.unique(src_nodes)

    # Aggregate attention per source node across layers and heads
    neighbor_attn = {}
    for src in unique_src:
        layer_attns = []
        for layer_idx, attn in enumerate(graph_attentions):
            # attn shape: (E, n_heads) — attention for each edge per head
            # Find edges from src → node_idx
            src_mask = (edge_index[0] == src) & (edge_index[1] == node_idx)
            edge_positions = torch.where(src_mask)[0]
            if len(edge_positions) > 0:
                # Average attention across heads for this edge
                edge_attn = attn[edge_positions].mean().item()
                layer_attns.append(edge_attn)
            else:
                layer_attns.append(0.0)

        neighbor_attn[int(src)] = {
            "per_layer": layer_attns,
            "mean": float(np.mean(layer_attns)),
        }

    # Sort by mean attention (descending)
    ranked = sorted(neighbor_attn.items(), key=lambda x: x[1]["mean"],
                    reverse=True)

    # Compute attention entropy (how focused vs. diffuse)
    attn_values = np.array([v["mean"] for _, v in ranked])
    attn_sum = attn_values.sum()
    if attn_sum > 0:
        attn_probs = attn_values / attn_sum
        attn_probs = attn_probs[attn_probs > 0]
        entropy = float(-np.sum(attn_probs * np.log(attn_probs)))
    else:
        entropy = 0.0

    # Build neighbor list with optional GT context
    neighbors = []
    for src, attn_info in ranked[:top_k]:
        entry = {
            "node_idx": src,
            "sv_label": idx_to_label.get(src) if idx_to_label else None,
            "mean_attention": attn_info["mean"],
            "per_layer_attention": attn_info["per_layer"],
        }
        # Add GT context if available
        if targets and idx_to_label:
            label = idx_to_label.get(src)
            if label and label in targets:
                t = targets[label]
                entry["y_cls"] = t["y_cls"]
                entry["y_dominant"] = t["y_dominant"]
                entry["centroid"] = t["centroid"].tolist()
        neighbors.append(entry)

    return {
        "node_idx": node_idx,
        "sv_label": idx_to_label.get(node_idx) if idx_to_label else None,
        "neighbors": neighbors,
        "attention_entropy": entropy,
        "total_incoming_edges": len(incoming_indices),
    }


# ── Patch-Level Attention Traces ─────────────────────────────────────


def patch_attention_trace(node_idx, patch_attentions, n_modalities=4,
                          n_patch=None, modality_names=None):
    n_patch = n_patch or GRAPH["n_patch"]
    modality_names = modality_names or GRAPH["modalities"]
    n_rows = n_patch * n_modalities

    if not patch_attentions:
        return {
            "node_idx": node_idx,
            "cls_to_patch_attention": np.zeros(n_rows),
            "per_patch_importance": np.zeros(n_patch),
            "per_modality_importance": np.zeros(n_modalities),
            "top_patches": [],
            "attention_layers": 0,
        }

    # Average [CLS]→patch attention across layers and heads
    # Attention shape: (B, n_heads, seq_len, seq_len)
    # seq_len = 1 (CLS) + n_rows (patches)
    # CLS is at position 0
    cls_attn_layers = []
    for attn in patch_attentions:
        # attn[node_idx]: (n_heads, seq_len, seq_len)
        # CLS → patches: row 0, columns 1:n_rows+1
        node_attn = attn[node_idx]  # (n_heads, seq_len, seq_len)
        cls_to_all = node_attn[:, 0, 1:]  # (n_heads, n_rows) — CLS to patches
        cls_attn_layers.append(cls_to_all.cpu().numpy())

    # Average across layers and heads
    avg_attn = np.mean(cls_attn_layers, axis=(0, 1))  # (n_rows,)

    # Aggregate per patch centroid (sum over modalities within each patch)
    per_patch = np.zeros(n_patch)
    for p in range(n_patch):
        start = p * n_modalities
        end = start + n_modalities
        per_patch[p] = avg_attn[start:end].sum()

    # Aggregate per modality (sum over patches within each modality)
    per_mod = np.zeros(n_modalities)
    for m in range(n_modalities):
        indices = [p * n_modalities + m for p in range(n_patch)]
        per_mod[m] = avg_attn[indices].sum()

    # Rank individual patch rows
    ranked_indices = np.argsort(avg_attn)[::-1]
    top_patches = []
    for idx in ranked_indices[:8]:  # top 8 rows
        patch_id = idx // n_modalities
        mod_id = idx % n_modalities
        top_patches.append({
            "row_idx": int(idx),
            "patch_id": int(patch_id),
            "modality_id": int(mod_id),
            "modality_name": modality_names[mod_id] if mod_id < len(modality_names) else f"mod_{mod_id}",
            "attention": float(avg_attn[idx]),
        })

    return {
        "node_idx": node_idx,
        "cls_to_patch_attention": avg_attn,
        "per_patch_importance": per_patch,
        "per_modality_importance": per_mod,
        "modality_names": modality_names,
        "top_patches": top_patches,
        "attention_layers": len(patch_attentions),
    }

### Level 3: Regression Refinement + Level 4: Task Divergence

In [ ]:
# ── Level 3: Regression Refinement Trace (adapted from Plan 1) ───────


def regression_refinement_trace(outputs, targets, retained_labels,
                                seg_features=None, idx_to_label=None,
                                edge_index=None, graph_attentions=None):
    N = len(retained_labels)
    y_reg = outputs["y_reg"].cpu().numpy()
    ensemble_var = outputs["ensemble_preds"].var(dim=-1).cpu().numpy()

    gt_reg = np.array([targets[retained_labels[i]]["y_reg"] for i in range(N)])
    gt_cls = np.array([targets[retained_labels[i]]["y_cls"] for i in range(N)])

    # Seg model prediction: derive tumor proportion proxy from seg_feat
    if seg_features is not None:
        seg_tumor_prob = np.array([
            1.0 - seg_features[retained_labels[i]]["seg_feat"][0]  # 1 - P(BG)
            for i in range(N)
        ])
    else:
        seg_tumor_prob = np.zeros(N)

    # Regression error
    mae = float(np.abs(y_reg - gt_reg).mean())
    r2_denom = np.var(gt_reg) * N
    r2 = float(1.0 - np.sum((y_reg - gt_reg)**2) / max(r2_denom, 1e-8))

    # Seg error
    seg_mae = float(np.abs(seg_tumor_prob - gt_reg).mean())

    # Find nodes where GNN substantially improves over seg model
    gnn_error = np.abs(y_reg - gt_reg)
    seg_error = np.abs(seg_tumor_prob - gt_reg)
    improvement = seg_error - gnn_error  # positive = GNN better

    # Corrections: GNN much better than seg
    correction_mask = improvement > 0.1
    # Degradations: GNN much worse than seg
    degradation_mask = improvement < -0.1

    corrections = []
    for i in range(N):
        if not correction_mask[i] and not degradation_mask[i]:
            continue

        label = retained_labels[i]
        entry = {
            "node_idx": i,
            "sv_label": label,
            "gt_reg": float(gt_reg[i]),
            "gt_cls": int(gt_cls[i]),
            "gnn_reg": float(y_reg[i]),
            "seg_tumor_prob": float(seg_tumor_prob[i]),
            "gnn_error": float(gnn_error[i]),
            "seg_error": float(seg_error[i]),
            "improvement": float(improvement[i]),
            "ensemble_var": float(ensemble_var[i]),
            "type": "correction" if correction_mask[i] else "degradation",
            "gt_dominant": targets[label]["y_dominant"],
            "centroid": targets[label]["centroid"].tolist(),
        }

        # Attention trace for corrections
        if (correction_mask[i] and edge_index is not None
                and graph_attentions is not None):
            trace = graph_attention_trace(
                i, edge_index, graph_attentions,
                targets=targets, idx_to_label=idx_to_label, top_k=3,
            )
            entry["attention_trace"] = trace["neighbors"]
            entry["attention_entropy"] = trace["attention_entropy"]

        corrections.append(entry)

    # Zero-shot Dice: threshold regression at τ to recover binary
    tau = GRAPH.get("tau", 0.15)
    gnn_binary = (y_reg > tau).astype(int)
    tp = ((gnn_binary == 1) & (gt_cls == 1)).sum()
    fp = ((gnn_binary == 1) & (gt_cls == 0)).sum()
    fn = ((gnn_binary == 0) & (gt_cls == 1)).sum()
    dice = float(2 * tp / max(2 * tp + fp + fn, 1))

    return {
        "total_nodes": N,
        "mae_gnn": mae,
        "mae_seg": seg_mae,
        "r2": r2,
        "zero_shot_dice": dice,
        "n_corrections": int(correction_mask.sum()),
        "n_degradations": int(degradation_mask.sum()),
        "mean_ensemble_var": float(ensemble_var.mean()),
        "corrections": corrections,
    }


# ── Level 4: Task-Specific Divergence ────────────────────────────────


def task_divergence_trace(outputs, targets, retained_labels,
                          seg_features=None, idx_to_label=None):
    N = len(retained_labels)
    y_reg = outputs["y_reg"].cpu().numpy()
    unc_prob = torch.sigmoid(outputs["unc_logits"]).cpu().numpy()

    # For edges, compute per-node dominant edge type
    edge_logits = outputs["edge_logits"]
    edge_index = outputs.get("_edge_index")  # injected by explain_case

    # Per-node incident edge analysis
    edge_preds = edge_logits.argmax(dim=-1).cpu().numpy() if edge_logits is not None else None

    node_edge_profile = {}
    if edge_preds is not None and edge_index is not None:
        ei = edge_index.cpu().numpy() if torch.is_tensor(edge_index) else edge_index
        for i in range(N):
            mask = (ei[0] == i) | (ei[1] == i)
            incident = edge_preds[mask]
            if len(incident) > 0:
                type_counts = np.bincount(incident, minlength=10)
                dominant_type = int(type_counts.argmax())
                has_nontrivial = int((incident > 0).sum())  # non-BG↔BG
            else:
                type_counts = np.zeros(10, dtype=int)
                dominant_type = 0
                has_nontrivial = 0
            node_edge_profile[i] = {
                "dominant_type": dominant_type,
                "has_nontrivial_edges": has_nontrivial,
                "type_counts": type_counts,
            }

    # Classify each node's divergence pattern
    tau = GRAPH.get("tau", 0.15)
    unc_threshold = 0.5
    node_divergences = []
    category_counts = {}

    for i in range(N):
        label = retained_labels[i]
        reg_high = y_reg[i] > tau
        unc_high = unc_prob[i] > unc_threshold
        ep = node_edge_profile.get(i, {})
        dom_type = ep.get("dominant_type", 0)
        has_nontrivial = ep.get("has_nontrivial_edges", 0) > 0

        # Classify divergence
        if reg_high and not unc_high and dom_type > 0:
            category = "AGREEMENT_CONFIDENT"
        elif not reg_high and not unc_high and dom_type == 0:
            category = "AGREEMENT_NEGATIVE"
        elif reg_high and unc_high:
            category = "DISAGREEMENT_REG_UNC"
        elif reg_high and dom_type == 0:
            category = "DISAGREEMENT_EDGE"
        elif has_nontrivial and unc_high:
            category = "UNCERTAIN_BOUNDARY"
        else:
            category = "MIXED"

        category_counts[category] = category_counts.get(category, 0) + 1

        gt = targets.get(label, {})
        entry = {
            "node_idx": i,
            "sv_label": label,
            "category": category,
            "y_reg": float(y_reg[i]),
            "unc_prob": float(unc_prob[i]),
            "dominant_edge_type": dom_type,
            "has_nontrivial_edges": has_nontrivial,
            "gt_cls": gt.get("y_cls", -1),
            "gt_reg": gt.get("y_reg", -1),
            "gt_dominant": gt.get("y_dominant", -1),
        }

        # Add seg context
        if seg_features and label in seg_features:
            sf = seg_features[label]
            entry["seg_pred"] = sf["seg_pred"]
            entry["seg_entropy"] = float(sf["seg_entropy"])

        node_divergences.append(entry)

    # Find the most clinically interesting nodes: DISAGREEMENT categories
    interesting = [n for n in node_divergences
                   if n["category"].startswith("DISAGREEMENT")
                   or n["category"] == "UNCERTAIN_BOUNDARY"]

    return {
        "total_nodes": N,
        "category_counts": category_counts,
        "interesting_nodes": interesting[:20],  # cap for readability
        "all_divergences": node_divergences,
    }

### Level 5: Uncertainty-Driven + Full Report

In [ ]:
# ── Level 5: Uncertainty-Driven Explanation──────────────────────────


def uncertainty_explanation_trace(outputs, targets, retained_labels,
                                  seg_features=None, idx_to_label=None,
                                  edge_index=None, graph_attentions=None,
                                  unc_threshold=0.6, top_k=10):
    N = len(retained_labels)
    unc_prob = torch.sigmoid(outputs["unc_logits"]).cpu().numpy()
    y_reg = outputs["y_reg"].cpu().numpy()
    ens_var = outputs["ensemble_preds"].var(dim=-1).cpu().numpy()

    # Find high-uncertainty nodes
    high_unc_indices = np.where(unc_prob > unc_threshold)[0]
    # Sort by uncertainty descending
    high_unc_indices = high_unc_indices[np.argsort(unc_prob[high_unc_indices])[::-1]]
    high_unc_indices = high_unc_indices[:top_k]

    explanations = []
    for i in high_unc_indices:
        label = retained_labels[i]
        gt = targets.get(label, {})

        entry = {
            "node_idx": int(i),
            "sv_label": label,
            "unc_prob": float(unc_prob[i]),
            "y_reg": float(y_reg[i]),
            "ensemble_var": float(ens_var[i]),
            "gt_reg": float(gt.get("y_reg", -1)),
            "gt_cls": int(gt.get("y_cls", -1)),
            "gt_dominant": int(gt.get("y_dominant", -1)),
            "centroid": gt.get("centroid", np.zeros(3)).tolist(),
        }

        # Was the seg model actually wrong?
        if seg_features and label in seg_features:
            sf = seg_features[label]
            seg_pred = sf["seg_pred"]
            gt_dom = gt.get("y_dominant", 0)
            seg_was_wrong = int(seg_pred != gt_dom)
            entry["seg_pred"] = seg_pred
            entry["seg_entropy"] = float(sf["seg_entropy"])
            entry["seg_was_actually_wrong"] = seg_was_wrong
            entry["seg_feat"] = sf["seg_feat"].tolist()

            # Does regression compensate?
            # If seg says BG but regression says tumor (y_reg > τ)
            tau = GRAPH.get("tau", 0.15)
            seg_says_bg = (seg_pred == 0)
            reg_says_tumor = (y_reg[i] > tau)
            entry["regression_compensates"] = bool(
                seg_was_wrong and seg_says_bg and reg_says_tumor
            )

        # Graph attention trace: what neighbors influence this uncertain node?
        if edge_index is not None and graph_attentions is not None:
            attn_trace = graph_attention_trace(
                int(i), edge_index, graph_attentions,
                targets=targets, idx_to_label=idx_to_label, top_k=5,
            )
            entry["attention_trace"] = attn_trace["neighbors"]
            entry["attention_entropy"] = attn_trace["attention_entropy"]

            # Neighbor consistency: are neighbors of uncertain node also uncertain?
            neighbor_unc = []
            for nb in attn_trace["neighbors"]:
                nb_idx = nb["node_idx"]
                if nb_idx < N:
                    neighbor_unc.append(float(unc_prob[nb_idx]))
            entry["neighbor_mean_unc"] = float(np.mean(neighbor_unc)) if neighbor_unc else 0.0

        explanations.append(entry)

    # Global uncertainty statistics
    # Correlation between ensemble variance and uncertainty probability
    if N > 1:
        corr = float(np.corrcoef(ens_var, unc_prob)[0, 1])
    else:
        corr = 0.0

    # How well does the uncertainty head detect actual seg errors?
    if seg_features is not None:
        gt_unc = np.array([
            1 if seg_features[retained_labels[i]]["seg_pred"]
            != targets[retained_labels[i]]["y_dominant"]
            else 0
            for i in range(N)
        ])
        # AUROC approximation: how well does unc_prob rank actual errors?
        from sklearn.metrics import roc_auc_score
        try:
            auroc = float(roc_auc_score(gt_unc, unc_prob))
        except (ValueError, ImportError):
            auroc = -1.0
        actual_error_rate = float(gt_unc.mean())
    else:
        auroc = -1.0
        actual_error_rate = -1.0

    return {
        "total_nodes": N,
        "n_high_uncertainty": len(high_unc_indices),
        "unc_threshold": unc_threshold,
        "mean_unc_prob": float(unc_prob.mean()),
        "ensemble_unc_correlation": corr,
        "auroc": auroc,
        "actual_seg_error_rate": actual_error_rate,
        "explanations": explanations,
    }


# ── Full Explanation Report (5 levels) ───────────────────────────────


def explain_case(model, result, device=None, top_k_nodes=10):
    # compute_laplacian_pe already defined above

    device = device or GRAPH.get("device", "cpu")
    retained = result["retained_labels"]
    N = len(retained)

    # Assemble tensors
    patch_list = [torch.from_numpy(result["patches"][l]) for l in retained]
    patch_batch = torch.stack(patch_list, dim=0).to(device)

    if result["seg_features"] is not None:
        seg_list = []
        for l in retained:
            sf = result["seg_features"][l]
            feat = np.concatenate([sf["seg_feat"], [sf["seg_entropy"]]])
            seg_list.append(torch.from_numpy(feat))
        seg_batch = torch.stack(seg_list, dim=0).float().to(device)
    else:
        seg_batch = torch.zeros(N, 5, device=device)

    ei = torch.from_numpy(result["edge_index"]).to(device)
    ea = torch.from_numpy(result["edge_attr"]).to(device)
    lpe = compute_laplacian_pe(result["edge_index"], N).to(device)

    # Forward with attention
    model.eval()
    with torch.no_grad():
        outputs, attn_dict = model(
            patch_batch, seg_batch, ei, ea,
            lap_pe=lpe, return_attention=True,
        )

    # Inject edge_index for task_divergence_trace
    outputs["_edge_index"] = ei

    # ── Level 1: Graph attention traces ──
    # Focus on tumor SVs and high-regression predictions
    tumor_indices = [i for i in range(N)
                     if result["targets"][retained[i]]["y_cls"] == 1]
    high_reg = outputs["y_reg"].cpu().argsort(descending=True)[:top_k_nodes].tolist()
    explain_indices = list(set(tumor_indices + high_reg))[:top_k_nodes]

    graph_traces = {}
    for idx in explain_indices:
        graph_traces[idx] = graph_attention_trace(
            idx, ei, attn_dict["graph"],
            targets=result["targets"],
            idx_to_label=result["idx_to_label"],
        )

    # ── Level 2: Patch attention traces ──
    patch_traces = {}
    if attn_dict["patch"]:
        for idx in explain_indices:
            patch_traces[idx] = patch_attention_trace(
                idx, attn_dict["patch"],
            )

    # ── Level 3: Regression refinement trace ──
    reg_trace = regression_refinement_trace(
        outputs, result["targets"], retained,
        seg_features=result["seg_features"],
        idx_to_label=result["idx_to_label"],
        edge_index=ei, graph_attentions=attn_dict["graph"],
    )

    # ── Level 4: Task divergence trace ──
    div_trace = task_divergence_trace(
        outputs, result["targets"], retained,
        seg_features=result["seg_features"],
        idx_to_label=result["idx_to_label"],
    )

    # ── Level 5: Uncertainty-driven explanation ──
    unc_trace = uncertainty_explanation_trace(
        outputs, result["targets"], retained,
        seg_features=result["seg_features"],
        idx_to_label=result["idx_to_label"],
        edge_index=ei, graph_attentions=attn_dict["graph"],
    )

    return {
        "case_id": result["case_id"],
        "graph_traces": graph_traces,
        "patch_traces": patch_traces,
        "regression_refinement": reg_trace,
        "task_divergence": div_trace,
        "uncertainty_explanation": unc_trace,
        "outputs": {k: v.cpu() if torch.is_tensor(v) else v
                    for k, v in outputs.items() if k != "_edge_index"},
    }

### Generate 5-Level Explainability Report

In [ ]:
if cases:
    test_result = all_graphs[0] if all_graphs else preprocess_case(cases[0], seg_prob_dir=str(seg_prob_dir))
    t0 = time.time()
    report = explain_case(model, test_result, device=DEVICE)
    print(f"Explanation generated in {time.time()-t0:.1f}s\n")

    # Level 1
    print("=" * 60)
    print("Level 1: Graph Attention Traces")
    print("=" * 60)
    for idx, trace in list(report["graph_traces"].items())[:3]:
        label = trace["sv_label"]
        gt = test_result["targets"].get(label, {})
        reg = float(report["outputs"]["y_reg"][idx])
        print(f"  Node {idx} (SV {label}): reg={reg:.3f}, GT={gt.get('y_reg','?'):.3f}, entropy={trace['attention_entropy']:.3f}")

    # Level 2
    print(f"\n{'='*60}\nLevel 2: Patch Attention\n{'='*60}")
    for idx, trace in list(report["patch_traces"].items())[:2]:
        print(f"  Node {idx}: ", end="")
        for m, name in enumerate(trace["modality_names"]):
            print(f"{name}={trace['per_modality_importance'][m]:.3f}", end="  ")
        print()

    # Level 3
    reg = report["regression_refinement"]
    print(f"\n{'='*60}\nLevel 3: Regression Refinement\n{'='*60}")
    print(f"  MAE(GNN)={reg['mae_gnn']:.4f}, MAE(Seg)={reg['mae_seg']:.4f}, R²={reg['r2']:.4f}")
    print(f"  Zero-shot Dice={reg['zero_shot_dice']:.4f}, Ensemble var={reg['mean_ensemble_var']:.6f}")

    # Level 4
    div = report["task_divergence"]
    print(f"\n{'='*60}\nLevel 4: Task Divergence\n{'='*60}")
    for cat, count in sorted(div["category_counts"].items()):
        print(f"  {cat}: {count}")

    # Level 5
    unc = report["uncertainty_explanation"]
    print(f"\n{'='*60}\nLevel 5: Uncertainty-Driven\n{'='*60}")
    print(f"  High-unc SVs: {unc['n_high_uncertainty']} (threshold={unc['unc_threshold']})")
    print(f"  Mean unc={unc['mean_unc_prob']:.4f}, Ensemble-unc corr={unc['ensemble_unc_correlation']:.4f}")
    print(f"  AUROC={unc['auroc']:.4f}")

    print(f"\nPlan 2 Explainability verified (5 levels) ✓")